# Building AI Agents with the Agno Framework

**A Complete, Step-by-Step Tutorial for Google Colab**

This notebook teaches you how to build production-style AI agents using **Agno** (formerly known as Phidata), a lightweight, high-performance Python framework for building agents with memory, knowledge, tools, and reasoning.

**Models used in this notebook:**
- Chat model: `gpt-4o-mini` (fast and inexpensive)
- Embedding model: `text-embedding-3-small` (for knowledge bases / RAG)

**What you will learn:**

| Section | Topic |
|---|---|
| 1 | Setup and installation |
| 2 | Your first agent |
| 3 | Personas: descriptions and instructions |
| 4 | Agents with built-in tools (web search, finance data) |
| 5 | Writing your own custom tools |
| 6 | Structured outputs with Pydantic |
| 7 | Memory, sessions, and chat history |
| 8 | Knowledge bases and RAG with embeddings |
| 9 | Multi-agent teams |
| 10 | Real-life project: Customer Support Agent |
| 11 | Real-life project: Financial Analyst Agent |
| 12 | Best practices and next steps |

**Prerequisites:** an OpenAI API key (https://platform.openai.com/api-keys). A GPU is not required; agents run on API calls.

## 1. Setup and Installation

We install Agno pinned to the stable 1.x series so every code cell in this notebook runs without breaking changes, plus the supporting libraries:

- `openai` - the OpenAI SDK used by Agno under the hood
- `duckduckgo-search` - free web search tool
- `yfinance` - free stock market data tool
- `lancedb` - an embedded vector database for our knowledge base (no server needed, perfect for Colab)
- `pypdf` - PDF parsing for document knowledge bases
- `sqlalchemy` - used by Agno's SQLite session storage

In [ ]:
%pip install -qU "agno>=1.7,<2.0" openai duckduckgo-search yfinance lancedb pypdf sqlalchemy

import agno
print("Agno version:", agno.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 995.0/995.0 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.7/58.7 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.6/405.6 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 107.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 20.6 MB/s eta 0:00:00
Agno version: 1.8.4


### 1.1 Configure your OpenAI API key

The cell below prompts you for your key and stores it in an environment variable. It is never printed to the notebook output.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("API key configured:", bool(os.environ.get("OPENAI_API_KEY")))

Enter your OpenAI API key: ··········
API key configured: True


## 2. Your First Agent

**What is an agent?** A plain LLM call takes text in and returns text out. An *agent* wraps a model with:

1. **Instructions** - a role and rules that shape its behavior
2. **Tools** - functions it can decide to call (search the web, query a database, do math)
3. **Memory** - the ability to remember previous turns of a conversation
4. **Knowledge** - documents it can search to ground its answers (RAG)

Agno's core abstraction is the `Agent` class. The minimal agent needs only a model.

In [25]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat

# The simplest possible agent: a model and nothing else
basic_agent = Agent(
    model=OpenAIChat(id="gpt-4o-mini"),
    markdown=True,          # render responses as Markdown
)

# print_response streams the answer directly into the notebook
basic_agent.print_response("Explain what an AI agent is in two sentences.", stream=True)

Output()

### 2.1 `print_response` vs `run`

- `agent.print_response(...)` prints a nicely formatted answer. Great for notebooks and demos.
- `agent.run(...)` returns a `RunResponse` object. Use this when you need the answer *as data* inside your program.

In [ ]:
response = basic_agent.run("Give me three creative names for a coffee shop in Tokyo.")

# The RunResponse object carries the content plus useful metadata
print(type(response))
print("-" * 60)
print(response.content)
print("-" * 60)
print("Metrics:", response.metrics)   # token usage and timing per model call

<class 'agno.run.response.RunResponse'>
------------------------------------------------------------
Here are three creative names for a coffee shop in Tokyo:

1. **Shibuya Brews**  
   A nod to the bustling Shibuya district, this name captures the vibrant energy of the area while emphasizing the shop's focus on high-quality coffee.

2. **Zen & Java**  
   Combining the calmness of Zen philosophy with the invigorating nature of coffee, this name evokes a serene environment perfect for relaxation and reflection.

3. **Kawaii Café**  
   Embracing the Japanese culture of cuteness, "Kawaii Café" suggests a cozy, inviting space where patrons can enjoy cute latte art and delightful pastries.
------------------------------------------------------------
Metrics: {'input_tokens': [39], 'output_tokens': [130], 'total_tokens': [169], 'audio_tokens': [0], 'input_audio_tokens': [0], 'output_audio_tokens': [0], 'cached_tokens': [0], 'cache_write_tokens': [0], 'reasoning_tokens': [0], 'prompt_tokens

## 3. Personas: Description and Instructions

Two parameters control an agent's behavior:

- `description` - *who the agent is* (one or two sentences, becomes part of the system prompt)
- `instructions` - *how it should behave* (a list of concrete rules)

This is the single highest-leverage place to improve agent quality. Be specific.

In [ ]:
travel_agent = Agent(
    model=OpenAIChat(id="gpt-4o-mini"),
    description="You are a seasoned travel consultant who specializes in budget travel across Asia.",
    instructions=[
        "Always give price estimates in USD.",
        "Recommend a maximum of 3 options so the traveler is not overwhelmed.",
        "For each option include: best season to visit, one hidden gem, and a rough daily budget.",
        "Be concise. Use tables where they improve readability.",
    ],
    markdown=True,
)

travel_agent.print_response("I have $1,500 and 10 days. Where should I go in Southeast Asia?", stream=True)

Output()

## 4. Agents with Tools

Tools turn a chatbot into an agent. The model *decides on its own* when to call a tool, receives the result, and uses it to answer.

Agno ships with 80+ pre-built toolkits (web search, finance, email, SQL, Python execution, and more). Here we use **DuckDuckGo** for free web search - no extra API key required.

Setting `show_tool_calls=True` prints each tool call in the output, which is extremely useful for learning and debugging.

> Note: DuckDuckGo occasionally rate-limits shared cloud IPs such as Colab. If a search cell fails, simply re-run it after a few seconds.

In [ ]:
!pip install ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 8.0 MB/s eta 0:00:00


In [ ]:
from agno.tools.duckduckgo import DuckDuckGoTools

web_agent = Agent(
    model=OpenAIChat(id="gpt-4o-mini"),
    tools=[DuckDuckGoTools()],
    description="You are a news research assistant.",
    instructions=[
        "Search the web before answering questions about current events.",
        "Always cite your sources with links at the end of the answer.",
    ],
    show_tool_calls=True,   # display which tools the agent calls
    markdown=True,
)

web_agent.print_response("What are the latest developments in solid-state batteries?", stream=True)

Output()

### 4.1 Finance tools

`YFinanceTools` gives the agent live stock market data through Yahoo Finance. Notice that we do not tell the agent *which* function to call - it reads the question, picks the right tool, and formats the result.

In [ ]:
from agno.tools.yfinance import YFinanceTools

finance_agent = Agent(
    model=OpenAIChat(id="gpt-4o-mini"),
    tools=[
        YFinanceTools(
            stock_price=True,
            company_info=True,
            analyst_recommendations=True,
        )
    ],
    description="You are a financial data assistant.",
    instructions=[
        "Use tables to display numeric data.",
        "Always state that this is information, not investment advice.",
    ],
    show_tool_calls=True,
    markdown=True,
)

finance_agent.print_response("Compare the current stock price and analyst recommendations for NVDA and AMD.", stream=True)

Output()

## 5. Writing Your Own Custom Tools

Any Python function can become a tool. The rules:

1. Add **type hints** to every parameter - the model uses them to build correct arguments.
2. Write a clear **docstring** - the model reads it to decide *when* to use the tool.
3. Return a **string** (or something easily convertible to one).

Below we simulate two real business functions: checking an order status and calculating shipping cost. In production these would call your real database or API.

In [ ]:
from agno.tools import tool

# A tiny fake database standing in for a real one
FAKE_ORDERS = {
    "ORD-1001": {"status": "Shipped",    "eta": "2026-07-08", "item": "Wireless Headphones"},
    "ORD-1002": {"status": "Processing", "eta": "2026-07-12", "item": "Mechanical Keyboard"},
    "ORD-1003": {"status": "Delivered",  "eta": "2026-06-30", "item": "USB-C Hub"},
}

@tool(show_result=True)
def get_order_status(order_id: str) -> str:
    """Look up the status of a customer order by its order ID (format: ORD-XXXX)."""
    order = FAKE_ORDERS.get(order_id.strip().upper())
    if not order:
        return f"No order found with ID {order_id}."
    return f"Order {order_id}: {order['item']} - status: {order['status']}, estimated arrival: {order['eta']}."

@tool(show_result=True)
def calculate_shipping_cost(weight_kg: float, express: bool = False) -> str:
    """Calculate the shipping cost for a package. weight_kg is the package weight in kilograms.
    Set express=True for 2-day express delivery."""
    base = 5.00 + weight_kg * 2.50
    if express:
        base *= 1.8
    return f"Shipping cost: ${base:.2f} ({'express 2-day' if express else 'standard 5-7 day'})."

orders_agent = Agent(
    model=OpenAIChat(id="gpt-4o-mini"),
    tools=[get_order_status, calculate_shipping_cost],
    description="You are an order support assistant for an electronics store.",
    instructions=["Always look up real data with your tools instead of guessing."],
    show_tool_calls=True,
    markdown=True,
)

orders_agent.print_response(
    "Where is my order ORD-1002? Also, how much would express shipping cost for a 3.5 kg package?",
     stream=True
)

Output()

## 6. Structured Outputs with Pydantic

Free-form text is hard to feed into other software. With `response_model` the agent returns a **validated Pydantic object** instead of prose - ideal for building APIs, filling databases, or driving UI components.

In [ ]:
from typing import List
from pydantic import BaseModel, Field

class JobPosting(BaseModel):
    title: str = Field(..., description="Job title")
    seniority: str = Field(..., description="Junior, Mid, Senior, or Lead")
    salary_range_usd: str = Field(..., description="Estimated yearly salary range in USD")
    key_skills: List[str] = Field(..., description="5 most important skills")
    one_line_pitch: str = Field(..., description="One sentence to attract candidates")

hr_agent = Agent(
    model=OpenAIChat(id="gpt-4o-mini"),
    description="You write structured job postings for a tech recruiting platform.",
    response_model=JobPosting,   # forces the output into this exact schema
)

result = hr_agent.run("Create a posting for a machine learning engineer working on recommendation systems.")

posting = result.content          # this is a JobPosting object, not a string
print(type(posting))
print("Title:   ", posting.title)
print("Level:   ", posting.seniority)
print("Salary:  ", posting.salary_range_usd)
print("Skills:  ", ", ".join(posting.key_skills))
print("Pitch:   ", posting.one_line_pitch)

<class '__main__.JobPosting'>
Title:    Machine Learning Engineer - Recommendation Systems
Level:    Mid
Salary:   $100,000 - $130,000
Skills:   Machine Learning, Python, Data Analysis, Collaborative Filtering, Deep Learning
Pitch:    Join our innovative team to build cutting-edge recommendation systems that enhance user experiences!


## 7. Memory, Sessions, and Chat History

By default each `run()` is stateless - the agent forgets everything. Two parameters fix that:

- `add_history_to_messages=True` - include previous turns in the prompt
- `storage=SqliteStorage(...)` - persist sessions to a SQLite file so conversations survive restarts

Each conversation is identified by a `session_id`. Different session IDs are completely separate conversations, which is how you support many users at once.

In [ ]:
import os
os.makedirs("tmp", exist_ok=True)

# Storage import (path differs slightly across 1.x minor versions; this handles both)
try:
    from agno.storage.sqlite import SqliteStorage
except ImportError:
    from agno.storage.agent.sqlite import SqliteAgentStorage as SqliteStorage

storage = SqliteStorage(table_name="agent_sessions", db_file="tmp/agents.db")

memory_agent = Agent(
    model=OpenAIChat(id="gpt-4o-mini"),
    storage=storage,
    session_id="user_alex_session_1",   # everything under this ID is one conversation
    add_history_to_messages=True,       # send prior turns back to the model
    num_history_responses=5,            # how many previous exchanges to include
    description="You are a friendly personal assistant.",
    markdown=True,
)

memory_agent.print_response("Hi, my name is Alex and I am training for a half marathon in October.", stream=True)

Output()

In [26]:
# Second turn: the agent recalls the name and the goal from the stored history
memory_agent.print_response("What is my name, and what am I training for?")

Output()

## 8. Knowledge Bases and RAG

**The problem:** models do not know your private documents (company policies, product manuals, internal wikis) and they hallucinate when asked about them.

**The solution - Retrieval-Augmented Generation (RAG):**

1. Your documents are split into chunks and converted into **embeddings** (numeric vectors) using `text-embedding-3-small`.
2. The vectors are stored in a **vector database** (we use LanceDB - it runs inside Colab, no server needed).
3. When the user asks a question, the agent **searches** the knowledge base for the most relevant chunks and answers *from those chunks*.

With `search_knowledge=True` Agno adds a `search_knowledge_base` tool to the agent, so retrieval happens automatically only when needed. This is called *Agentic RAG*.

In [ ]:
from agno.document.base import Document
from agno.knowledge.document import DocumentKnowledgeBase
from agno.vectordb.lancedb import LanceDb, SearchType
from agno.embedder.openai import OpenAIEmbedder

# 1. The embedder: text-embedding-3-small (1536-dimensional vectors)
embedder = OpenAIEmbedder(id="text-embedding-3-small", dimensions=1536)

# 2. The vector database (a local folder inside Colab)
vector_db = LanceDb(
    table_name="company_policies",
    uri="tmp/lancedb",
    search_type=SearchType.vector,
    embedder=embedder,
)

# 3. Some internal company documents (in real life: load PDFs, websites, Notion, etc.)
policy_docs = [
    Document(content=(
        "Return Policy: TechNova offers a 30-day return window on all electronics. "
        "Items must be in original packaging. Refunds are issued to the original payment "
        "method within 5-7 business days. Opened software and gift cards are non-refundable."
    )),
    Document(content=(
        "Warranty Policy: All TechNova products include a 12-month limited warranty covering "
        "manufacturing defects. Accidental damage is not covered unless the customer purchased "
        "TechNova Care+, which extends coverage to 24 months and includes two accidental damage repairs."
    )),
    Document(content=(
        "Shipping Policy: Standard shipping is free on orders over $50 and takes 5-7 business days. "
        "Express shipping costs $14.99 and delivers within 2 business days. TechNova ships to the US, "
        "Canada, and the EU. Orders placed before 2 PM EST ship the same day."
    )),
]

knowledge_base = DocumentKnowledgeBase(documents=policy_docs, vector_db=vector_db)

# 4. Load = chunk + embed + store. recreate=True rebuilds from scratch (use False after the first run).
knowledge_base.load(recreate=True)
print("Knowledge base loaded.")

INFO Creating table: company_policies

INFO Dropping collection

INFO Creating collection

INFO Creating table: company_policies

INFO Loading knowledge base

INFO Added 3 documents to knowledge base

Knowledge base loaded.


In [ ]:
rag_agent = Agent(
    model=OpenAIChat(id="gpt-4o-mini"),
    knowledge=knowledge_base,
    search_knowledge=True,    # gives the agent a search_knowledge_base tool
    description="You are a support agent for TechNova, an electronics retailer.",
    instructions=[
        "Answer ONLY from the knowledge base. If the answer is not there, say you do not know.",
        "Quote the relevant policy when helpful.",
    ],
    show_tool_calls=True,
    markdown=True,
)

rag_agent.print_response("I bought a laptop 3 weeks ago and it stopped working. Can I return it, and does the warranty cover it?", stream=True)

Output()

INFO Found 3 documents

INFO Found 3 documents

### 8.1 Loading knowledge from a PDF URL (optional)

The same pattern works with real files. Agno provides ready-made knowledge base classes such as `PDFUrlKnowledgeBase`, `TextKnowledgeBase`, `WebsiteKnowledgeBase`, and `CSVKnowledgeBase`. Example with a public PDF:

In [ ]:
from agno.knowledge.pdf_url import PDFUrlKnowledgeBase

pdf_kb = PDFUrlKnowledgeBase(
    urls=["https://agno-public.s3.amazonaws.com/recipes/ThaiRecipes.pdf"],
    vector_db=LanceDb(
        table_name="thai_recipes",
        uri="tmp/lancedb",
        search_type=SearchType.vector,
        embedder=OpenAIEmbedder(id="text-embedding-3-small", dimensions=1536),
    ),
)
pdf_kb.load(recreate=False)   # only embeds on the first run

recipe_agent = Agent(
    model=OpenAIChat(id="gpt-4o-mini"),
    knowledge=pdf_kb,
    search_knowledge=True,
    instructions=["Answer only from the recipe knowledge base."],
    show_tool_calls=True,
    markdown=True,
)

recipe_agent.print_response("How do I make Pad Thai? List the key ingredients.", stream=True)

INFO Creating table: thai_recipes

INFO Loading knowledge base

INFO Reading: https://agno-public.s3.amazonaws.com/recipes/ThaiRecipes.pdf

INFO Added 14 documents to knowledge base

Output()

INFO Found 5 documents

## 9. Multi-Agent Teams

Complex tasks benefit from **specialist agents working together**. Agno's `Team` class supports three modes:

- `coordinate` - a team leader breaks the task down, delegates to members, and synthesizes the final answer (most common)
- `route` - the leader forwards the request to the single best member
- `collaborate` - all members answer and the leader merges their outputs

Below, a **web researcher** and a **financial analyst** cooperate on a market question.

In [ ]:
from agno.team import Team

researcher = Agent(
    name="Web Researcher",
    role="Finds recent news and qualitative information on the web",
    model=OpenAIChat(id="gpt-4o-mini"),
    tools=[DuckDuckGoTools()],
    instructions=["Always include source links."],
)

analyst = Agent(
    name="Financial Analyst",
    role="Retrieves and interprets stock market data",
    model=OpenAIChat(id="gpt-4o-mini"),
    tools=[YFinanceTools(stock_price=True, analyst_recommendations=True, company_info=True)],
    instructions=["Present numbers in tables."],
)

market_team = Team(
    name="Market Research Team",
    mode="coordinate",
    model=OpenAIChat(id="gpt-4o-mini"),   # the team leader model
    members=[researcher, analyst],
    instructions=[
        "Combine qualitative news with quantitative market data.",
        "Finish with a short balanced summary. This is information, not investment advice.",
    ],
    show_tool_calls=True,
    markdown=True,
)

market_team.print_response("Give me a brief investment overview of Tesla: recent news plus current stock data.", stream=True)

Output()

## 10. Real-Life Project: Complete Customer Support Agent

Now we combine **everything** into one production-style agent for the fictional retailer TechNova:

- **Knowledge base** - answers policy questions from the documents in Section 8
- **Custom tools** - looks up live order data
- **Session memory** - remembers the customer across the whole conversation
- **Clear instructions** - polite tone, escalation rules, no guessing

This is the exact architecture behind most real support bots.

In [ ]:
support_agent = Agent(
    model=OpenAIChat(id="gpt-4o-mini"),
    description="You are 'Nova', the customer support agent for TechNova electronics.",
    instructions=[
        "Be warm, professional, and concise.",
        "For policy questions, search the knowledge base and quote the policy.",
        "For order questions, use the order lookup tool.",
        "Never invent order details or policies. If unsure, offer to escalate to a human agent.",
        "Address the customer by name once you know it.",
    ],
    knowledge=knowledge_base,           # Section 8 policies
    search_knowledge=True,
    tools=[get_order_status, calculate_shipping_cost],   # Section 5 tools
    storage=storage,                    # Section 7 storage
    session_id="customer_maria_ticket_4821",
    add_history_to_messages=True,
    num_history_responses=5,
    show_tool_calls=True,
    markdown=True,
)

support_agent.print_response(
    "Hi, I'm Maria. My order ORD-1001 hasn't arrived yet. Also, what's your return policy if I don't like it?", stream=True
)

Output()

INFO Found 3 documents

In [ ]:
# Follow-up turn: memory (name, order) + knowledge base working together
support_agent.print_response("Thanks. And if it breaks in 6 months, am I covered?", stream=True)

Output()

## 11. Real-Life Project: Financial Analyst Report Generator

A second common pattern: an agent that produces a **structured report** as data. This combines live tools with a Pydantic `response_model`, so the result can go straight into a dashboard, PDF generator, or database.

In [ ]:
class StockReport(BaseModel):
    ticker: str
    company_name: str
    current_price: str = Field(..., description="Current price with currency symbol")
    analyst_consensus: str = Field(..., description="Overall analyst sentiment, e.g. Buy / Hold / Sell")
    strengths: List[str] = Field(..., description="2-3 key strengths")
    risks: List[str] = Field(..., description="2-3 key risks")
    summary: str = Field(..., description="Three-sentence overview. Not investment advice.")

report_agent = Agent(
    model=OpenAIChat(id="gpt-4o-mini"),
    tools=[YFinanceTools(stock_price=True, company_info=True, analyst_recommendations=True)],
    description="You produce concise, factual stock reports from live market data.",
    response_model=StockReport,
)

report = report_agent.run("Create a report for Microsoft (MSFT).").content

print("Ticker:   ", report.ticker)
print("Company:  ", report.company_name)
print("Price:    ", report.current_price)
print("Consensus:", report.analyst_consensus)
print("Strengths:", *["\n  - " + s for s in report.strengths])
print("Risks:    ", *["\n  - " + r for r in report.risks])
print("\nSummary:\n", report.summary)

Ticker:    MSFT
Company:   Microsoft Corporation
Price:     390.49 USD
Consensus: Strong Buy
Strengths: 
  - Strong revenue growth of 18.3% 
  - High gross margins at 68.3% 
  - Diverse product offerings across various segments
Risks:     
  - Exposure to competitive technology market 
  - Potential regulatory scrutiny 
  - Dependence on enterprise subscriptions

Summary:
 Microsoft Corporation, founded in 1975 and headquartered in Redmond, WA, is a leading technology company specializing in software, services, and devices. It operates through segments such as Productivity and Business Processes, Intelligent Cloud, and Personal Computing, catering to both consumers and enterprises. The company's current stock price is 390.49 USD with an analyst consensus rating of Strong Buy.


## 12. Best Practices and Next Steps

**Best practices learned in this notebook:**

1. **Instructions are your main lever.** Specific, numbered rules beat long paragraphs.
2. **Keep `show_tool_calls=True` during development** so you can see exactly what the agent is doing; turn it off in production output.
3. **Ground factual answers in a knowledge base** and instruct the agent to admit when the answer is not there - this is the single best defense against hallucination.
4. **Use `response_model`** whenever the output feeds another system; never parse free text with regex.
5. **One agent, one job.** When a task needs several skills, prefer a `Team` of focused agents over one bloated agent.
6. **Pin your dependency versions** (as we did with `agno>=1.7,<2.0`) so tutorials and production code do not break when new major versions ship.
7. **Watch costs.** `gpt-4o-mini` and `text-embedding-3-small` are among the cheapest options; `response.metrics` shows token usage per call.

**Where to go next:**

- Official documentation: https://docs.agno.com
- GitHub (examples and cookbook): https://github.com/agno-agi/agno
- Explore further features: reasoning tools, agentic workflows, image/audio inputs, the Agent Playground UI, and monitoring at https://app.agno.com

**Troubleshooting in Colab:**

- `DuckDuckGo rate limit` errors: wait a few seconds and re-run the cell.
- `RuntimeError: asyncio` issues: restart the runtime (Runtime > Restart session) and re-run from the top.
- Knowledge base re-embedding on every run: set `recreate=False` in `knowledge_base.load()` after the first successful load.